# 08 · Las cinco sensibilidades fijadas en el Avance 2
**Grupo 1 · Estadística Bayesiana · Entrega final**  
Jean Paul Rojas Rojas · Daniel Estiven Pacheco Villagrande · Diego Andrés Cascante Camacho

**Qué hace.** Repite M1 con la prior de κ centrada en 10 y en 300, con franjas de tres horas y con solo siniestros con muertos, y compara el orden de M2 con 2020 (Tabla 3 del informe).

**Entradas.** `datos/siniestros_limpios.csv`; `salidas/m1_resumen.npz` y `m2_resumen.npz`.

**Salidas.** `salidas/resultados/08_sensibilidades.json`.

*Respaldo por etapas: el notebook principal es `../Grupo1_Bayesiana_completo.ipynb`, que reúne estas ocho etapas.* Orden de ejecución y trazabilidad completa: `../README.md`.

In [1]:
# Configuración: funciona en Google Colab y en un computador
# En Colab: suba la carpeta "notebooks" a su Drive. Si el buscador automático no la encuentra,
# escriba su ruta en BASE_COLAB, p. ej. '/content/drive/MyDrive/Grupo1_EntregaFinal_Bayesiana/notebooks'
BASE_COLAB = ''
import os, sys, glob
from pathlib import Path
EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not BASE_COLAB:
        cand = glob.glob('/content/drive/MyDrive/**/notebooks/comun.py', recursive=True)
        assert cand, 'No encontré la carpeta notebooks en su Drive: escriba su ruta en BASE_COLAB'
        BASE_COLAB = os.path.dirname(cand[0])
    BASE = Path(BASE_COLAB)
else:
    # el notebook puede estar en notebooks/ o en notebooks/por_etapas/
    BASE = Path.cwd() if (Path.cwd() / 'comun.py').exists() else Path.cwd().parent
os.chdir(BASE)
sys.path.insert(0, str(BASE))
import numpy, pandas, scipy, matplotlib
print('Carpeta:', BASE)
print('Python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| scipy', scipy.__version__, '| matplotlib', matplotlib.__version__)

Carpeta: C:\Users\dddan\Downloads\Pensamiento - Bayesiana\4. Estadística Bayesiana\Proyecto\4. Entrega final\notebooks
Python 3.12.10 | numpy 2.3.5 | pandas 2.3.3 | scipy 1.18.1 | matplotlib 3.10.7


In [2]:
from comun import *
from mcmc import correr_m1
a1, a2 = np.load(SALIDAS / 'm1_resumen.npz'), np.load(SALIDAS / 'm2_resumen.npz')
p1, p2 = a1['media'], a2['media']
R = {}

In [3]:
# ---------------------------------------------------------------------------
# 11. Cinco sensibilidades fijadas en el Avance 2
# ---------------------------------------------------------------------------
sens = {}
for nom, rt in (("kappa centrada en 10", 2 / 10), ("kappa centrada en 300", 2 / 300)):
    o = correr_m1(n_tr, y_tr, rt_k=rt, seed=SEED + 11)
    mt = metricas(o["theta"].reshape(-1, L * F).mean(0))
    sens[nom] = dict(spearman=mt["spearman"], brier=mt["brier"], top10=mt["top10"],
                     kappa=float(o["kappa"].mean()),
                     rho_vs_M1=float(st.spearmanr(o["theta"].reshape(-1, L * F).mean(0)[eleg], p1[eleg])[0]))
# franjas de tres horas
c3_tr = celdas(tr, "franja3h", FR3)
c3_te = celdas(te, "franja3h", FR3)
o = correr_m1(c3_tr.n.values, c3_tr.y.values, seed=SEED + 12)
p3 = o["theta"].reshape(-1, len(c3_tr)).mean(0)
el3 = (c3_tr.n.values >= 30) & (c3_te.n.values >= 30)
ob3 = np.where(c3_te.n > 0, c3_te.y / np.maximum(c3_te.n, 1), np.nan)
cr3 = np.where(c3_tr.n > 0, c3_tr.y / np.maximum(c3_tr.n, 1), 0.33)
i3 = np.where(el3)[0]
sens["franjas de 3 horas (160 celdas)"] = dict(
    spearman=float(st.spearmanr(p3[el3], ob3[el3])[0]),
    spearman_cruda=float(st.spearmanr(cr3[el3], ob3[el3])[0]),
    top10=len(set(i3[np.argsort(-p3[i3])[:10]]) & set(i3[np.argsort(-ob3[i3])[:10]])),
    celdas_eleg=int(el3.sum()),
    brier=float(metricas(p3, c3_te.n.values, c3_te.y.values, el3)["brier"]),
    brier_cruda=float(metricas(cr3, c3_te.n.values, c3_te.y.values, el3)["brier"]))
# solo siniestros con muertos (menos sensible al subregistro de choques leves)
cm_tr, cm_te = celdas(tr, y="con_muertos"), celdas(te, y="con_muertos")
o = correr_m1(cm_tr.n.values, cm_tr.y.values, a_m=1.0, b_m=30.0, seed=SEED + 13)
pm = o["theta"].reshape(-1, L * F).mean(0)
obm = np.where(cm_te.n > 0, cm_te.y / np.maximum(cm_te.n, 1), np.nan)
sens["solo siniestros con muertos"] = dict(
    spearman=float(st.spearmanr(pm[eleg], obm[eleg])[0]),
    spearman_cruda=float(st.spearmanr(np.where(cm_tr.n > 0, cm_tr.y / np.maximum(cm_tr.n, 1), 0)[eleg], obm[eleg])[0]),
    rho_vs_graves=float(st.spearmanr(pm[eleg], p2[eleg])[0]),
    tasa_muertos_tr=float(cm_tr.y.sum() / cm_tr.n.sum()),
    m=float(o["m"].mean()), kappa=float(o["kappa"].mean()),
    top10_comun_con_M2=len(set(idx_e[np.argsort(-pm[idx_e])[:10]]) & set(idx_e[np.argsort(-p2[idx_e])[:10]])),
    top10=[f"{c_tr.localidad[i]} | {c_tr.franja[i]} ({100 * pm[i]:.1f} %)" for i in idx_e[np.argsort(-pm[idx_e])[:10]]],
    top10_sur=int(es_sur[idx_e[np.argsort(-pm[idx_e])[:10]]].sum()),
    top10_noche=int(es_noche[idx_e[np.argsort(-pm[idx_e])[:10]]].sum()))
# 2020 (pandemia): ¿se sostiene el orden?
n20, y20 = c_20.n.values, c_20.y.values
el20 = (n_tr >= 30) & (n20 >= 30)
ob20 = np.where(n20 > 0, y20 / np.maximum(n20, 1), np.nan)
i20 = np.where(el20)[0]
sens["2020 (ruptura)"] = dict(
    spearman_M2=float(st.spearmanr(p2[el20], ob20[el20])[0]),
    spearman_cruda=float(st.spearmanr(p_cruda_tr[el20], ob20[el20])[0]),
    spearman_conteo=float(st.spearmanr(y_tr[el20], ob20[el20])[0]),
    top10_M2=len(set(i20[np.argsort(-p2[i20])[:10]]) & set(i20[np.argsort(-ob20[i20])[:10]])),
    celdas_eleg=int(el20.sum()))
R["sensibilidades"] = sens
print(pd.DataFrame({k: {c: v for c, v in d.items() if not isinstance(v, list)} for k, d in sens.items()}).T.round(3).to_string())
print('Top 10 solo muertos:', sens['solo siniestros con muertos']['top10'])
guardar_json(R, '08_sensibilidades.json')

                                 spearman  brier  top10    kappa  rho_vs_M1  spearman_cruda  celdas_eleg  brier_cruda  rho_vs_graves  tasa_muertos_tr      m  top10_comun_con_M2  top10_sur  top10_noche  spearman_M2  spearman_conteo  top10_M2
kappa centrada en 10                0.923  0.222    6.0   21.338        1.0             NaN          NaN          NaN            NaN              NaN    NaN                 NaN        NaN          NaN          NaN              NaN       NaN
kappa centrada en 300               0.923  0.222    6.0   23.933        1.0             NaN          NaN          NaN            NaN              NaN    NaN                 NaN        NaN          NaN          NaN              NaN       NaN
franjas de 3 horas (160 celdas)     0.906  0.221    5.0      NaN        NaN           0.902        149.0        0.221            NaN              NaN    NaN                 NaN        NaN          NaN          NaN              NaN       NaN
solo siniestros con muertos         